# 05. Feature Selection, Encoding, Preprocessing

## Objective

The feature engineering stage aims to prepare and construct optimal features for use in the employee attrition prediction model.

The process involves the following steps:

1. Split the target variable and predictor variables.
2. Identifying and removing irrelevant features, such as identifiers and constant features.
3. Determining which features to maintained based on the results of EDA and statistical analysis.
4. Encoding categorical features.
5. Scaling numerical features where necessary.
6. Build a preprocessing pipeline to avoid data leakage.
7. Generating a feature dataset ready for the modeling stage.

The output of this stage consists of features prepared for the machine learning model.

## Import Library

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", None)

sns.set_theme(style="whitegrid")

## Load Dataset

In [2]:
DATA_PATH = Path("../data/clean/clean_employee_attrition.csv")

data = pd.read_csv(DATA_PATH)

data.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,AgeGroup,TenureGroup,MonthlyIncomeBracket,OverallSatisfactionIndex
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5,35-44,Established (6-10 yrs),High,2.00
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7,45-54,Established (6-10 yrs),High,3.00
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0,35-44,New (0-2 yrs),Low,3.00
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0,25-34,Established (6-10 yrs),Low,3.25
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2,25-34,New (0-2 yrs),Medium,2.50


## Split Target and Features

In [3]:
TARGET = "Attrition"

In [4]:
X = data.drop(columns=[TARGET])
y = data[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1470, 38)
y shape: (1470,)


In [5]:
y.value_counts(normalize=True) * 100

Attrition
No     83.877551
Yes    16.122449
Name: proportion, dtype: float64

## Convert Target Variable

For machine learning, Attrition needs to be converted into a binary target.

In [6]:
y = y.map({
    "No": 0,
    "Yes": 1
})
y

0       1
1       0
2       1
3       0
4       0
       ..
1465    0
1466    0
1467    0
1468    0
1469    0
Name: Attrition, Length: 1470, dtype: int64

In [7]:
print(y.value_counts())

Attrition
0    1233
1     237
Name: count, dtype: int64


In [8]:
y.name = "Attrition"

## Identify Identifier Features

In dataset, there are several features that logically do not provide useful predictive information.

1. EmployeeNumber
2. EmployeeCount
3. StandardHours
4. Over18

In [9]:
identifier_features = [
    "EmployeeNumber"
]

constant_features = [
    "EmployeeCount",
    "StandardHours",
    "Over18"
]

In [10]:
features_to_drop = [
    col for col in (
        identifier_features + constant_features
    )
    if col in X.columns
]

features_to_drop

['EmployeeNumber', 'EmployeeCount', 'StandardHours', 'Over18']

In [11]:
X = X.drop(columns=features_to_drop)

print("Removed features:")
print(features_to_drop)

print("\nNew shape:", X.shape)

Removed features:
['EmployeeNumber', 'EmployeeCount', 'StandardHours', 'Over18']

New shape: (1470, 34)


It possesses an identifier value or a constant value in all observations, and thus does not help the model differentiate between employees attrition and those who do not.

## Feature Selection Based on Statistical Analysis

Based on the results of the previous statistical tests, we select the significant features.

In [12]:
significant_categorical = [
     'OverTime',
     'JobRole',
     'MonthlyIncomeBracket'
]

In [13]:
significant_numeric = [
     'TotalWorkingYears',
     'MonthlyIncome',
     'YearsAtCompany',
     'JobLevel',
     'YearsInCurrentRole',
     'YearsWithCurrManager',
     'Age'
]

In [14]:
selected_features = significant_categorical + significant_numeric
selected_features

['OverTime',
 'JobRole',
 'MonthlyIncomeBracket',
 'TotalWorkingYears',
 'MonthlyIncome',
 'YearsAtCompany',
 'JobLevel',
 'YearsInCurrentRole',
 'YearsWithCurrManager',
 'Age']

In [15]:
selected_features = [
    feature
    for feature in selected_features
    if feature in X.columns
]

X = X[selected_features]

In [16]:
print("Selected features:")
print(X.columns.tolist())

print("\nNumber of selected features:", X.shape[1])

Selected features:
['OverTime', 'JobRole', 'MonthlyIncomeBracket', 'TotalWorkingYears', 'MonthlyIncome', 'YearsAtCompany', 'JobLevel', 'YearsInCurrentRole', 'YearsWithCurrManager', 'Age']

Number of selected features: 10


In [17]:
X

,OverTime,JobRole,MonthlyIncomeBracket,TotalWorkingYears,MonthlyIncome,YearsAtCompany,JobLevel,YearsInCurrentRole,YearsWithCurrManager,Age
0,Yes,Sales Executive,High,8,5993,6,2,4,5,41
1,No,Research Scientist,High,10,5130,10,2,7,7,49
2,Yes,Laboratory Technician,Low,7,2090,0,1,0,0,37
3,Yes,Research Scientist,Low,8,2909,8,1,7,0,33
4,No,Laboratory Technician,Medium,6,3468,2,1,2,2,27
...,...,...,...,...,...,...,...,...,...,...
1465,No,Laboratory Technician,Low,17,2571,5,2,2,3,36
1466,No,Healthcare Representative,Very High,9,9991,7,3,7,7,39
1467,Yes,Manufacturing Director,High,6,6142,6,2,2,3,27
1468,No,Sales Executive,High,17,5390,9,2,6,8,49


In [18]:
y

0       1
1       0
2       1
3       0
4       0
       ..
1465    0
1466    0
1467    0
1468    0
1469    0
Name: Attrition, Length: 1470, dtype: int64

## Train-Test Split

In [19]:
from sklearn.model_selection import train_test_split

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Re-identify Numerical and Categorical Features

In [21]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

In [22]:
print("Numerical:", numeric_features)
print()
print("Categorical:", categorical_features)

Numerical: ['TotalWorkingYears', 'MonthlyIncome', 'YearsAtCompany', 'JobLevel', 'YearsInCurrentRole', 'YearsWithCurrManager', 'Age']

Categorical: ['OverTime', 'JobRole', 'MonthlyIncomeBracket']


## Numerical Preprocessing

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [24]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

## Categorical Preprocessing

In [25]:
from sklearn.preprocessing import OneHotEncoder

In [26]:
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first"))
    ]
)

## Combine Preprocessing

In [27]:
from sklearn.compose import ColumnTransformer

In [28]:
preprocessor = ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [29]:
 #                 X
 #                 │
 #        ┌────────┴────────┐
 #        │                 │
 #   Numerical         Categorical
 #        │                 │
 #        ▼                 ▼
 # StandardScaler      One-Hot Encoder
 #        │                 │
 #        └────────┬────────┘
 #                 ▼
 #          Final Features

## Fit Preprocessor pada Training Data

In [30]:
X_train_transformed = preprocessor.fit_transform(X_train)

for testing

In [31]:
X_test_transformed = preprocessor.transform(X_test)

## Check Transformation Result

In [32]:
print(
    "Original X_train:",
    X_train.shape
)

print(
    "Transformed X_train:",
    X_train_transformed.shape
)

Original X_train: (1176, 10)
Transformed X_train: (1176, 19)


In [33]:
print(
    "Original X_test:",
    X_test.shape
)

print(
    "Transformed X_test:",
    X_test_transformed.shape
)

Original X_test: (294, 10)
Transformed X_test: (294, 19)


The number of columns may increase because categorical features run into One-Hot Encoding.

## Get Feature Names

In [34]:
feature_names = preprocessor.get_feature_names_out()

print("Jumlah final features:", len(feature_names))

Jumlah final features: 19


In [35]:
feature_names

array(['num__TotalWorkingYears', 'num__MonthlyIncome',
       'num__YearsAtCompany', 'num__JobLevel', 'num__YearsInCurrentRole',
       'num__YearsWithCurrManager', 'num__Age', 'cat__OverTime_Yes',
       'cat__JobRole_Human Resources',
       'cat__JobRole_Laboratory Technician', 'cat__JobRole_Manager',
       'cat__JobRole_Manufacturing Director',
       'cat__JobRole_Research Director',
       'cat__JobRole_Research Scientist', 'cat__JobRole_Sales Executive',
       'cat__JobRole_Sales Representative',
       'cat__MonthlyIncomeBracket_Low',
       'cat__MonthlyIncomeBracket_Medium',
       'cat__MonthlyIncomeBracket_Very High'], dtype=object)

## Convert to DataFrame

Convert the preprocessing results into a DataFrame.

In [36]:
X_train_final = pd.DataFrame(
    X_train_transformed.toarray()
    if hasattr(X_train_transformed, "toarray")
    else X_train_transformed,
    
    columns=feature_names,
    
    index=X_train.index
)

In [37]:
X_train_final.head()

,num__TotalWorkingYears,num__MonthlyIncome,num__YearsAtCompany,num__JobLevel,num__YearsInCurrentRole,num__YearsWithCurrManager,num__Age,cat__OverTime_Yes,cat__JobRole_Human Resources,cat__JobRole_Laboratory Technician,cat__JobRole_Manager,cat__JobRole_Manufacturing Director,cat__JobRole_Research Director,cat__JobRole_Research Scientist,cat__JobRole_Sales Executive,cat__JobRole_Sales Representative,cat__MonthlyIncomeBracket_Low,cat__MonthlyIncomeBracket_Medium,cat__MonthlyIncomeBracket_Very High
1194,2.261482,2.026752,-0.665706,1.762189,-0.625365,-0.616406,1.090194,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
128,-1.072675,-0.864408,-0.830071,-0.986265,-0.905635,-0.897047,-1.634828,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
810,1.492061,2.347706,0.813578,1.762189,1.336527,1.348076,0.981193,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
478,-0.559727,-0.956202,-0.008246,-0.986265,-0.064824,0.506155,-1.307825,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
491,-0.175017,-0.185956,0.156119,-0.070114,0.775986,0.786795,0.654191,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [38]:
X_test_final = pd.DataFrame(
    X_test_transformed.toarray()
    if hasattr(X_test_transformed, "toarray")
    else X_test_transformed,
    
    columns=feature_names,
    
    index=X_test.index
)

In [39]:
X_test_final.head()

,num__TotalWorkingYears,num__MonthlyIncome,num__YearsAtCompany,num__JobLevel,num__YearsInCurrentRole,num__YearsWithCurrManager,num__Age,cat__OverTime_Yes,cat__JobRole_Human Resources,cat__JobRole_Laboratory Technician,cat__JobRole_Manager,cat__JobRole_Manufacturing Director,cat__JobRole_Research Director,cat__JobRole_Research Scientist,cat__JobRole_Sales Executive,cat__JobRole_Sales Representative,cat__MonthlyIncomeBracket_Low,cat__MonthlyIncomeBracket_Medium,cat__MonthlyIncomeBracket_Very High
1061,-1.329148,-0.969745,-0.994436,-0.986265,-1.185905,-1.177687,-1.416826,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
891,-0.175017,-0.974474,0.484849,-0.986265,0.215446,0.786795,0.763191,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
456,-0.175017,1.077650,-0.336976,0.846038,-0.064824,-0.897047,-0.653820,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
922,1.876772,2.718533,2.950322,2.678340,1.336527,2.470637,0.763191,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
69,-1.200911,-0.678457,-0.994436,-0.986265,-1.185905,-1.177687,-0.108815,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


## Final Feature Validation

In [40]:
print(
    "Missing values:",
    X_train_final.isnull().sum().sum()
)

print(
    "Missing values:",
    X_test_final.isnull().sum().sum()
)

Missing values: 0
Missing values: 0


In [41]:
print(
    "Infinite values:",
    np.isinf(X_train_final).sum().sum()
)
print(
    "Infinite values:",
    np.isinf(X_test_final).sum().sum()
)

Infinite values: 0
Infinite values: 0


In [42]:
print("X_train:", X_train_final.shape)
print("X_test :", X_test_final.shape)

X_train: (1176, 19)
X_test : (294, 19)


## Check Scaling

To ensure the StandardScaler works:

In [43]:
scaled_numeric_features = [
    f"num__{feature}"
    for feature in numeric_features
]

X_train_final[scaled_numeric_features].describe().T

,count,mean,std,min,25%,50%,75%,max
num__TotalWorkingYears,1176.0,-7.288199e-17,1.000425,-1.457385,-0.687964,-0.175017,0.466167,3.672087
num__MonthlyIncome,1176.0,-7.930164e-17,1.000425,-1.189876,-0.773045,-0.330955,0.403390,2.886856
num__YearsAtCompany,1176.0,-1.586033e-17,1.000425,-1.158801,-0.665706,-0.336976,0.484849,4.922701
num__JobLevel,1176.0,7.703588e-17,1.000425,-0.986265,-0.986265,-0.070114,0.846038,2.678340
num__YearsInCurrentRole,1176.0,5.588878e-17,1.000425,-1.185905,-0.625365,-0.345095,0.775986,3.578689
num__YearsWithCurrManager,1176.0,7.854639e-17,1.000425,-1.177687,-0.616406,-0.335766,0.786795,3.593199
num__Age,1176.0,-4.229421e-17,1.000425,-2.070831,-0.762821,-0.108815,0.654191,2.507205


## Save Train dan Test Features

In [44]:
OUTPUT_DIR = Path("../data/processed")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [45]:
train_final = X_train_final.copy()
train_final["Attrition"] = y_train

test_final = X_test_final.copy()
test_final["Attrition"] = y_test

In [46]:
train_final.to_csv(
    OUTPUT_DIR / "train_features.csv",
    index=False
)

test_final.to_csv(
    OUTPUT_DIR / "test_features.csv",
    index=False
)

## Save Preprocessor

In [47]:
import joblib

In [48]:
joblib.dump(
    preprocessor,
    OUTPUT_DIR / "preprocessor.pkl"
)

['..\\data\\processed\\preprocessor.pkl']

## Summary

The following steps were carried out during the Feature Engineering stage:

1. Loaded the dataset resulting from Data Cleaning.
2. Split the target variable `Attrition` from the predictor variables.
3. Converted `Attrition` into a binary format:
   - 0 = No
   - 1 = Yes
4. Removed uninformative identifier and constant features.
5. Selected features based on EDA and Statistical Analysis results.
6. Split numerical and categorical features.
7. Performed a train-test split.
8. Preprocessed numerical features using median imputation and StandardScaler.
10. Preprocessed categorical features using most-frequent imputation and One-Hot Encoding.
11. Used ColumnTransformer to combine preprocessing steps.
12. Avoid data leakage by fitting the preprocessing only on the training data.
13. Validated the preprocessing results.
14. Saved the feature engineering results and the preprocessing pipeline.

Output:

- `train_features.csv`
- `test_features.csv`
- `preprocessor.pkl`

The dataset is ready for use in the `06_modeling.ipynb` stage.